# Day 20 – Confidence Intervals
## Measuring Uncertainty Around Your Data

**30 Days of Data Analytics – M.V. Krishnakumari**

A point estimate gives us a number. A confidence interval communicates uncertainty around that estimate.

This notebook uses a public-grievance resolution-time example.

## Learning Objectives
- Understand confidence intervals
- Distinguish point and interval estimates
- Calculate standard error and margin of error
- Calculate a 95% confidence interval for a mean
- Calculate a confidence interval for a proportion
- Understand sample size and confidence level effects
- Distinguish statistical significance, practical significance, precision and accuracy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. Business Question

Suppose a department reports:

**Average grievance resolution time = 48 hours**

The analyst asks:

**How precisely has this average been estimated?**

A confidence interval provides a range associated with the estimation procedure.

In [ ]:
resolution_time = np.random.normal(
    loc=48, scale=12, size=300
)
resolution_time = np.maximum(resolution_time, 0)

n = len(resolution_time)
mean_resolution = np.mean(resolution_time)
sample_std = np.std(resolution_time, ddof=1)

print(f"Observations: {n}")
print(f"Mean: {mean_resolution:.2f} hours")
print(f"Sample SD: {sample_std:.2f} hours")

## 2. Standard Error

In [ ]:
standard_error = sample_std / np.sqrt(n)
print(f"Standard Error: {standard_error:.4f}")

## 3. 95% Confidence Interval for the Mean

Because the population standard deviation is unknown, we use the t-distribution.

In [ ]:
confidence_level = 0.95

ci = stats.t.interval(
    confidence_level,
    df=n - 1,
    loc=mean_resolution,
    scale=standard_error
)

lower, upper = ci

print(f"Mean: {mean_resolution:.2f} hours")
print(f"95% CI: {lower:.2f} to {upper:.2f} hours")

## 4. Margin of Error

In [ ]:
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha / 2, df=n - 1)
margin_of_error = t_critical * standard_error

print(f"Margin of Error: ±{margin_of_error:.2f} hours")

## 5. Visualize the Estimate

In [ ]:
plt.figure(figsize=(9, 3))
plt.errorbar(
    mean_resolution, 0,
    xerr=[[mean_resolution - lower], [upper - mean_resolution]],
    fmt="o", capsize=8
)
plt.axvline(mean_resolution, linestyle="--")
plt.xlabel("Resolution Time (Hours)")
plt.yticks([])
plt.title("Mean Resolution Time with 95% Confidence Interval")
plt.grid(axis="x", alpha=0.25)
plt.show()

## 6. Compare Confidence Levels

In [ ]:
for level in [0.90, 0.95, 0.99]:
    interval = stats.t.interval(
        level, df=n-1,
        loc=mean_resolution,
        scale=standard_error
    )
    print(
        f"{int(level*100)}% CI: "
        f"{interval[0]:.2f} to {interval[1]:.2f} hours"
    )

## 7. Effect of Sample Size

Larger samples generally reduce the standard error and produce narrower confidence intervals, assuming comparable data quality and sampling conditions.

In [ ]:
for sample_size in [30, 100, 300, 1000]:
    sample = np.random.normal(48, 12, sample_size)
    sample = np.maximum(sample, 0)

    m = sample.mean()
    s = sample.std(ddof=1)
    se = s / np.sqrt(sample_size)

    interval = stats.t.interval(
        0.95,
        df=sample_size - 1,
        loc=m,
        scale=se
    )

    print(
        f"n={sample_size:4d} | "
        f"95% CI width={interval[1]-interval[0]:.2f} hours"
    )

## 8. Confidence Interval for a Proportion

Example: 720 of 1,000 complaints were resolved within SLA.

In [ ]:
successes = 720
n_sla = 1000
proportion = successes / n_sla

print(f"SLA Compliance: {proportion:.2%}")

### Wilson 95% Confidence Interval

In [ ]:
z = stats.norm.ppf(0.975)

denominator = 1 + z**2 / n_sla
centre = (proportion + z**2/(2*n_sla)) / denominator

half_width = (
    z * np.sqrt(
        proportion*(1-proportion)/n_sla
        + z**2/(4*n_sla**2)
    ) / denominator
)

wilson_lower = centre - half_width
wilson_upper = centre + half_width

print(f"95% Wilson CI: {wilson_lower:.2%} to {wilson_upper:.2%}")

## 9. Confidence Interval for a Difference

This connects directly to **Day 19 – Hypothesis Testing**. We compare resolution times before and after a process change.

In [ ]:
np.random.seed(42)

before = np.random.normal(52, 12, 300)
after = np.random.normal(47, 12, 300)

before = np.maximum(before, 0)
after = np.maximum(after, 0)

result = stats.ttest_ind(
    before, after, equal_var=False
)

difference = before.mean() - after.mean()
ci_difference = result.confidence_interval(0.95)

print(f"Estimated difference: {difference:.2f} hours")
print(
    f"95% CI for difference: "
    f"{ci_difference.low:.2f} to {ci_difference.high:.2f} hours"
)
print(f"P-value: {result.pvalue:.6f}")

## 10. Statistical vs Practical Significance

A statistically significant result may still be operationally unimportant.

For public grievance analytics, consider:
- SLA targets
- Complaint volume
- Complaint severity
- Staffing
- Operational cost
- Seasonal workload

**Statistical significance ≠ practical significance**.

## 11. Precision vs Accuracy

**Precision:** How tightly repeated estimates cluster.

**Accuracy:** How close an estimate is to the true value.

A narrow confidence interval communicates statistical precision; it does not guarantee absence of bias.

**Precision ≠ Accuracy**

## 12. Confidence Interval vs Prediction Interval

**Confidence interval:** Estimates a population parameter such as the population mean.

**Prediction interval:** Estimates a range in which a future individual observation may fall.

Prediction intervals are generally wider because they include individual-level variation.

## 13. Public Governance Applications

Confidence intervals can support estimates of:
- Average grievance resolution time
- SLA compliance
- Citizen satisfaction
- Property-tax collection rate
- Waste collection efficiency
- Service response time
- Infrastructure repair time
- Citizen-service adoption rate

## 14. Analyst Checklist

1. What parameter are we estimating?
2. How was the sample selected?
3. Is the sample representative?
4. What confidence level is appropriate?
5. How wide is the interval?
6. Are there potential sources of bias?
7. Is the estimate sufficiently precise for the decision?
8. Is the result operationally meaningful?

## Key Takeaways

- A point estimate is not the whole story.
- Confidence intervals communicate uncertainty.
- Larger samples generally improve precision.
- Higher confidence levels generally produce wider intervals.
- Precision is not accuracy.
- Confidence intervals complement hypothesis testing.
- Statistical significance and practical significance are different.
- Sampling quality and bias remain critical.

### Final Thought

**Averages tell us where the data is. Confidence intervals tell us how precisely we have estimated it.**

**Observation → Estimation → Uncertainty → Evidence → Decision**

### Next
**Day 21 – Correlation & Regression**

GitHub Repository:
https://github.com/krishnakumarimv/30-Days-Data-Analytics-Cookbook